In [1]:
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np

In [2]:
train_dataset = pd.read_csv("../datasets/train_dataset.csv")
test_dataset = pd.read_csv("../datasets/test_dataset.csv")

In [3]:
X_train = train_dataset.drop(
  columns=[
    'property_id',
    'date',
    'revenue',
    'occupancy_rate',
    'Unnamed: 1'
  ]
)

y_train = train_dataset['occupancy_rate']

X_test = test_dataset.drop(
  columns=[
    'property_id',
    'date',
    'revenue',
    'occupancy_rate',
    'Unnamed: 1'
  ]
)

y_test = test_dataset['occupancy_rate']

X_train.columns = (
    X_train.columns
    .str.replace(' ', '_')
    .str.replace(r'[^A-Za-z0-9_]', '', regex=True)
)

X_test.columns = (
    X_test.columns
    .str.replace(' ', '_')
    .str.replace(r'[^A-Za-z0-9_]', '', regex=True)
)

train_valid_idx = X_train.dropna().index

X_train = X_train.loc[train_valid_idx]
y_train = y_train.loc[train_valid_idx]

test_valid_idx = X_test.dropna().index

X_test = X_test.loc[test_valid_idx]
y_test = y_test.loc[test_valid_idx]

In [4]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 199500 entries, 1 to 199999
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   location_score              199500 non-null  int64  
 1   amenities_score             199500 non-null  int64  
 2   rating                      199500 non-null  float64
 3   base_price                  199500 non-null  float64
 4   month                       199500 non-null  int64  
 5   weekend                     199500 non-null  int64  
 6   holiday                     199500 non-null  int64  
 7   demand                      199500 non-null  float64
 8   competitor_price            199500 non-null  float64
 9   nearby_event                199500 non-null  int64  
 10  market_trend                199500 non-null  float64
 11  final_price                 199500 non-null  float64
 12  season_Monsoon              199500 non-null  int64  
 13  season_Summer               19

In [5]:
model = LGBMRegressor()
model.fit(X_train,y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003656 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2231
[LightGBM] [Info] Number of data points in the train set: 199500, number of used features: 30
[LightGBM] [Info] Start training from score 0.653315


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [6]:
preds = np.array(model.predict(X_test))

In [7]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_true=y_test,y_pred=preds)
mse = mean_squared_error(y_true=y_test,y_pred=preds)
r2_scr = r2_score(y_true=y_test,y_pred=preds)

In [8]:
print(f"Mean Aboslute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"R2 Score: {r2_scr}")

Mean Aboslute Error: 0.013852915309551263
Mean Squared Error: 0.000383712138190566
R2 Score: 0.9936731907644808


In [9]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
})

importance_df = importance_df.sort_values(
    'importance',
    ascending=False
)

print(importance_df.head(15))

                       feature  importance
8             competitor_price        1239
11                 final_price        1027
2                       rating         239
1              amenities_score         136
3                   base_price         136
0               location_score         124
7                       demand          64
10                market_trend          14
26                 price_lag_1           8
29        rolling_30_day_price           6
23  property_type_Luxury_suite           4
28        rolling_7_day_demand           2
25             occupancy_lag_1           1
5                      weekend           0
13               season_Summer           0


In [ ]:
def predict

property_id
P0       [960.71]
P1       [957.14]
P10     [2930.36]
P100    [1007.14]
P101     [858.93]
          ...    
P95     [2992.86]
P96      [941.07]
P97     [2985.71]
P98     [1992.86]
P99      [926.79]
Name: base_price, Length: 500, dtype: object
